In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

cli_directory = Path(
    r"C:\Users\maxkr\.vscode\extensions\databricks.databricks-2.14.1-win32-x64\bin"
)

assert cli_directory.exists(), f"CLI-Ordner nicht gefunden: {cli_directory}"

os.environ["PATH"] = (
    str(cli_directory)
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("CLI:", shutil.which("databricks"))
print("Profil:", os.getenv("DATABRICKS_CONFIG_PROFILE"))
print("Serverless:", os.getenv("DATABRICKS_SERVERLESS_COMPUTE_ID"))

CLI: C:\Users\maxkr\.vscode\extensions\databricks.databricks-2.14.1-win32-x64\bin\databricks.EXE
Profil: DEFAULT
Serverless: auto


In [7]:
from databricks.connect import DatabricksSession

spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

print("Spark-Verbindung hergestellt")

Spark-Verbindung hergestellt


### Check der Datensätze im Schema

In [ ]:
for catalog in ["workspace", "main"]:
    print(f"\nSchemas in {catalog}:")
    spark.sql(f"SHOW SCHEMAS IN {catalog}").show(100, truncate=False)



Schemas in workspace:
+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+


Schemas in main:
+-------------------+
|databaseName       |
+-------------------+
|dbdemos_retail_c360|
|default            |
|information_schema |
+-------------------+



In [12]:
tables_df = spark.sql("""
    SHOW TABLES IN main.dbdemos_retail_c360
""")

tables_df.show(100, truncate=False)

+-------------------+------------------------------+-----------+
|database           |tableName                     |isTemporary|
+-------------------+------------------------------+-----------+
|dbdemos_retail_c360|dbdemos_retail_c360_event_logs|false      |
+-------------------+------------------------------+-----------+



Die Datens#tze sind als csv filesabgelegt

In [13]:
users_path = "/Volumes/main/dbdemos_retail_c360/c360/users"
orders_path = "/Volumes/main/dbdemos_retail_c360/c360/orders"
events_path = "/Volumes/main/dbdemos_retail_c360/c360/events"

users_df = spark.read.json(users_path)

orders_df = spark.read.json(orders_path)

events_df = spark.read.csv(
    events_path,
    header=True,
    inferSchema=True
)

## Users Dataframe

In [17]:
users_df.show()

+--------------------+---------+------+-----+-------+-------------------+--------------------+-----------+------+--------------------+-------------------+---------+
|             address|age_group| canal|churn|country|      creation_date|               email|  firstname|gender|                  id| last_activity_date| lastname|
+--------------------+---------+------+-----+-------+-------------------+--------------------+-----------+------+--------------------+-------------------+---------+
|Unit 0526 Box 041...|      7.0|WEBAPP| true|  SPAIN|03-06-2014 00:00:00|pittmanlynn@jorda...|      Kevin|   1.0|4822f0b4-76bc-49c...|06-07-2023 05:12:45|   Torres|
|5375 Dunn Centers...|      2.0|WEBAPP| true|    USA|08-09-2021 00:00:00|sandovalgerald@ol...|     Eugene|   1.0|4b04e0a8-5bb4-42e...|06-04-2023 12:19:27|   Martin|
|4600 Richardson D...|      7.0|MOBILE|false|  SPAIN|08-12-2021 00:00:00|  ybriggs@pierce.org|     Andrea|   1.0|c941b1c2-501a-43e...|06-06-2023 20:54:21|   Pierce|
|99916 Fle

In [37]:
from pyspark.sql import functions as F

print(f"Churn distribution:\n{users_df.groupBy('churn').count().show()}")
print(f"Distinct IDs:\n{users_df.select(
    F.countDistinct("id").alias("distinct_ids")
).first()["distinct_ids"]}")
print(f"Distinct emails:\n{users_df.select(
    F.countDistinct("email").alias("distinct_emails")
).first()["distinct_emails"]}")
print(f"Countries:\n{users_df.select(
    F.countDistinct("country").alias("distinct_countries")
).first()["distinct_countries"]}")
print(f"Countries:\n{users_df.select("country").distinct().show(truncate=False)}")

+-----+-----+
|churn|count|
+-----+-----+
|false|23749|
| true|45130|
+-----+-----+

Churn distribution:
None
Distinct IDs:
68879
Distinct emails:
660
Countries:
3
+-------+
|country|
+-------+
|USA    |
|FR     |
|SPAIN  |
+-------+

Countries:
None


## Orders Dataframe

In [18]:
orders_df.show()

+------+--------------------+----------+-------------------+--------------------+
|amount|                  id|item_count|   transaction_date|             user_id|
+------+--------------------+----------+-------------------+--------------------+
|  28.0|9462a4e4-7a08-43f...|       2.0|06-04-2023 07:52:21|dc0f2d81-3962-4de...|
|  60.0|b4d05008-dbe9-45e...|       2.0|06-06-2023 09:16:20|7d681dab-5d34-4f7...|
|  39.0|40df7b39-70ab-4af...|       1.0|06-03-2023 08:02:04|acd6ea8c-42f0-48c...|
|  70.0|aa7c221a-12ad-4ea...|       2.0|06-04-2023 22:29:23|dcc97a2c-8359-498...|
|  27.0|224f8d0c-c991-45c...|       1.0|06-05-2023 04:37:13|b4dd00c9-b0fb-478...|
| 105.0|f805d313-dc8d-4d3...|       3.0|06-01-2023 21:58:28|c1e49eed-2b21-4fe...|
|  63.0|de6173d8-9fe0-470...|       3.0|06-02-2023 05:33:33|5f5bab1d-b08b-400...|
|  63.0|153b35e7-7120-4ac...|       3.0|06-05-2023 11:04:34|77b4f313-5032-485...|
|  50.0|0da1c284-4801-452...|       2.0|06-07-2023 00:57:17|5805a1aa-c8cb-42b...|
|  42.0|e58a036a

## Events Dataframe

In [19]:
events_df.show()

+--------------------+--------------------+--------+-------------------+------+--------------------+--------------------+
|             user_id|            event_id|platform|               date|action|          session_id|                 url|
+--------------------+--------------------+--------+-------------------+------+--------------------+--------------------+
|1d196d48-da64-43e...|203ca493-f4de-43a...|     ios|06-02-2023 04:03:58|  view|203ca493-f4de-43a...|https://databrick...|
|34120c72-87ab-405...|                NULL|     ios|06-07-2023 05:12:45|  view|                NULL|https://databrick...|
|262ef5fc-9597-4f6...|ff24ca0d-f6b9-453...|     ios|06-04-2023 16:55:26|  view|ff24ca0d-f6b9-453...|https://databrick...|
|3e62f2e2-99b0-4d6...|c23bb14d-d79d-459...|   other|06-01-2023 10:42:53| click|c23bb14d-d79d-459...|https://databrick...|
|0291a59d-2df3-472...|7c084652-fc61-454...|     ios|06-02-2023 01:46:00|  view|7c084652-fc61-454...|https://databrick...|
|8bf96a02-6b15-40d...|96